# Cleaning The Dataset
- identifying the important features and converting them to usable data structure (string to one hot encoding)

In [25]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.model_selection import KFold,train_test_split

In [2]:
data = pd.read_csv("dataset/job_placement.csv")

In [3]:
data.head()

,id,name,gender,age,degree,stream,college_name,placement_status,salary,gpa,years_of_experience
0,1,John Doe,Male,25,Bachelor's,Computer Science,Harvard University,Placed,60000,3.7,2.0
1,2,Jane Smith,Female,24,Bachelor's,Electrical Engineering,Massachusetts Institute of Technology,Placed,65000,3.6,1.0
2,3,Michael Johnson,Male,26,Bachelor's,Mechanical Engineering,Stanford University,Placed,58000,3.8,3.0
3,4,Emily Davis,Female,23,Bachelor's,Information Technology,Yale University,Not Placed,0,3.5,2.0
4,5,David Brown,Male,24,Bachelor's,Computer Science,Princeton University,Placed,62000,3.9,2.0


In [4]:
data['college_name'].value_counts().head()

college_name
University of California--Berkeley          43
University of Michigan--Ann Arbor           43
University of Virginia                      43
University of Illinois--Urbana-Champaign    43
University of Colorado--Boulder             43
Name: count, dtype: int64

In [5]:
# conversion of target column to numeric binary classification
def convert_target(x):
    if x.lower() == "placed":
        return 1
    else:
        return 0

data['target'] = data['placement_status'].apply(convert_target)

In [6]:
# Multiplying the GPA to 2 because in colleges we have gpa out of 10
data['gpa'] = data['gpa']*2

# Analysis of the dataset
- the dataset contains a column with stream which we can convert to numerical by label encoding (as its unique)
- the dataset also contains a column with salary after placement which we can convert to simple binary column by keeping the value as one where salary>mean_of_salary
- the dataset also contains various college name for which i am gonna convert that column to frequency reputation , that is frequency of people placed through a college over totatl number of students to get the reputation.

In [20]:
stream_encoder = LabelEncoder()
stream_encoder.fit(data['stream'])
data['stream_encoded']=stream_encoder.transform(data['stream'])

In [8]:
mean_of_salary = data['salary'].mean()
def check(x):
    if x>=mean_of_salary.round(-2):
        return 1
    else:
        return 0
data[f'salary>{mean_of_salary.round(-2)}'] = data['salary'].apply(check)

In [ ]:
# target enc is how good the college is placement wise and freq enc is how common is this college

def target_encode_with_smoothing(df, col, target, n_splits=5, smoothing=10):
    global_mean = df[target].mean()
    encoded = pd.Series(index=df.index, dtype=float)
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        
        agg = train_fold.groupby(col)[target].agg(['mean', 'count'])
        
        smoother = 1 / (1 + np.exp(-(agg['count'] - smoothing)))
        agg['smoothed'] = smoother * agg['mean'] + (1 - smoother) * global_mean
        
        encoded.iloc[val_idx] = df.iloc[val_idx][col].map(agg['smoothed'])
    
    encoded.fillna(global_mean, inplace=True)
    return encoded

def frequency_encode(df, col):
    freq = df[col].value_counts() / len(df)
    return df[col].map(freq)

data['college_target_enc'] = target_encode_with_smoothing(
    data,
    col='college_name',
    target='target',
    smoothing=10
)

data['college_freq_enc'] = frequency_encode(data, 'college_name')

# Drop original column (optional)
data.drop(columns=['college_name'], inplace=True)

In [21]:
degree_encoder = LabelEncoder()
degree_encoder.fit(data['stream'])
data['stream_encoded'] = degree_encoder.transform(data['stream'])

In [22]:
data.head()

,id,name,gender,age,degree,stream,placement_status,salary,gpa,years_of_experience,target,stream_encoded,salary>52500.0,college_target_enc,college_freq_enc
0,1,John Doe,Male,25,Bachelor's,Computer Science,Placed,60000,7.4,2.0,1,0,1,0.814286,0.001429
1,2,Jane Smith,Female,24,Bachelor's,Electrical Engineering,Placed,65000,7.2,1.0,1,1,1,0.814286,0.001429
2,3,Michael Johnson,Male,26,Bachelor's,Mechanical Engineering,Placed,58000,7.6,3.0,1,4,1,0.814286,0.001429
3,4,Emily Davis,Female,23,Bachelor's,Information Technology,Not Placed,0,7.0,2.0,0,3,0,0.814286,0.001429
4,5,David Brown,Male,24,Bachelor's,Computer Science,Placed,62000,7.8,2.0,1,0,1,0.814286,0.001429


In [23]:
train_data = data[['age','gpa','years_of_experience','target','stream_encoded','salary>52500.0','college_target_enc']]

Index(['age', 'gpa', 'years_of_experience', 'target', 'stream_encoded',
       'salary>52500.0', 'college_target_enc'],
      dtype='str')

In [28]:
x_train,x_test,y_train,y_test = train_test_split(train_data[['age', 'gpa', 'years_of_experience','stream_encoded','salary>52500.0', 'college_target_enc']],train_data['target'],test_size=.2)